# Praktikum W5S2: K-Means Clustering

Di praktikum ini kita akan coba mengelompokkan pelanggan mal berdasarkan kemiripan karakteristik mereka, pakai algoritma **K-Means**. Bedanya dengan KNN atau Naive Bayes yang kita pelajari sebelumnya, K-Means ini **tidak butuh label/target** — kita cuma kasih data mentah, lalu algoritma sendiri yang mencari kelompok-kelompok yang mirip satu sama lain. Makanya K-Means termasuk *unsupervised learning*.

Dataset yang dipakai: `Mall_Customers.csv` (taruh satu folder dengan notebook ini sebelum dijalankan).

### 1. Memuat dan mengecek data

Langkah pertama ya seperti biasa, buka datanya dulu dan lihat-lihat isinya. Kita intip 5 baris pertama pakai `head()` supaya tahu kolom apa saja yang ada.

In [ ]:
import pandas as pd

df = pd.read_csv('Mall_Customers.csv')
df.head()

Sebelum lanjut, selalu baik untuk cek apakah ada data yang kosong/hilang. Kalau ada data kosong dan tidak ditangani, bisa bikin error atau hasil clustering jadi aneh nantinya.

In [ ]:
# cek missing values
df.isnull().sum()

Hasilnya semua kolom nilainya 0, artinya datanya bersih, tidak ada yang bolong. Kita bisa langsung lanjut tanpa perlu bersih-bersih data dulu.

Sekarang kita intip ringkasan statistiknya pakai `describe()` ini berguna untuk melihat gambaran umum seperti rata-rata, nilai minimum dan maksimum tiap kolom numerik.

In [ ]:
df.describe()

Dari sini kelihatan misalnya `Annual Income` itu rentangnya dari 15 sampai 137 (dalam satuan ribuan dolar), sedangkan `Spending Score` dari 1 sampai 99. Perbedaan skala kayak gini penting dicatat, karena nanti bakal berpengaruh ke K-Means (akan kita bahas di bagian scaling).

### 2. Fokus ke dua fitur dulu: Income dan Spending Score

Supaya lebih gampang dipahami dan divisualisasikan, kita mulai dari dua fitur saja: `Annual Income` dan `Spending Score`. Nanti baru kita coba tambah fitur usia di bagian selanjutnya.

In [ ]:
dataset = df.iloc[:, 3:]
dataset.head()

Kita simpan juga masing-masing kolom ke variabel `x` dan `y` biar gampang dipakai buat plotting.

In [ ]:
x = df.iloc[:, 3]
y = df.iloc[:, 4]

Sebelum dikelompokkan, coba kita lihat dulu sebaran datanya mentah-mentah lewat scatter plot. Ini penting supaya kita punya gambaran: kira-kira datanya memang membentuk gerombolan-gerombolan atau tidak.

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(x, y)
plt.show()

Kalau diperhatikan, titik-titiknya memang sudah kelihatan mengelompok dengan sendirinya ada yang numpuk di tengah, ada juga beberapa gerombolan kecil di pojok-pojok grafik. Ini pertanda bagus karena artinya data ini memang cocok untuk dikelompokkan.

### 3. Menyamakan skala data dan mencari jumlah klaster yang pas

Sebelum masuk ke K-Means, kita perlu melakukan **scaling**. Kenapa? Karena K-Means bekerja berdasarkan jarak antar titik data, dan kalau satu fitur punya angka yang jauh lebih besar dari fitur lain (misalnya Income 0–137 vs Spending Score 0–99), fitur yang angkanya lebih besar bisa "mendominasi" perhitungan jarak padahal belum tentu itu yang paling penting. Makanya kita pakai `MinMaxScaler` supaya semua fitur punya rentang yang sama, yaitu 0 sampai 1.

Setelah itu, kita juga perlu menentukan: sebenarnya mau dikelompokkan jadi berapa klaster sih datanya? Nah, cara umum buat jawab ini namanya **Elbow Method**. Caranya, kita coba-coba jumlah klaster dari 1 sampai 10, lalu lihat nilai **WCSS** (Within-Cluster Sum of Squares) untuk tiap percobaan. WCSS ini intinya mengukur seberapa rapat titik-titik dalam satu klaster terhadap pusat klasternya — makin kecil, makin rapat (makin bagus).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

# Scaling data dengan MinMaxScaler
scaler = MinMaxScaler()

# Mengambil kolom X (index 3) dan Y (index 4) dari DataFrame
X = pd.DataFrame({'Annual Income (k$)': x, 'Spending Score (1-100)': y})
X_scaled = scaler.fit_transform(X)

# Menentukan jumlah klaster optimal menggunakan Elbow Method
wcss = []
for i in range(1, 11):  # Coba k dari 1 sampai 10
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Membuat plot Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='-')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS (Within-Cluster Sum of Squares)')
plt.grid(True)
plt.show()

Nah, dari grafiknya biasanya kelihatan garisnya menurun tajam di awal, lalu mulai landai setelah titik tertentu bentuknya kayak siku tangan, makanya disebut "elbow" (siku). Titik siku inilah yang kita anggap sebagai jumlah klaster paling pas, karena setelah titik itu, menambah klaster lagi tidak terlalu banyak membantu mengurangi WCSS, cuma bikin model makin rumit tanpa manfaat besar.

### 4. Melatih model dan lihat hasilnya

Dari grafik Elbow di atas, titik siku terlihat ada di sekitar **k = 5**, jadi itu yang kita pakai.

In [ ]:
# dari elbow method, nilai k optimal untuk data ini adalah 5
kmeans = KMeans(n_clusters=5)
kmeans.fit(X)

# Plot hasil klaster
plt.scatter(x, y, c=kmeans.labels_)
plt.show()

Sekarang hasilnya jauh lebih jelas titik-titik yang tadinya cuma kelihatan "mengelompok sembarang" sekarang sudah dikasih warna sesuai klasternya masing-masing. Kalau diperhatikan, lima kelompok ini punya ciri khas yang cukup masuk akal buat dunia nyata, misalnya:

- Ada kelompok dengan income rendah tapi spending tinggi (mungkin orang yang suka boros meski gajinya pas-pasan)
- Ada kelompok income tinggi tapi spending rendah (orang hemat meski kaya)
- Ada kelompok income dan spending sama-sama tinggi (pelanggan "premium" yang paling potensial buat ditarget promosi)
- Dan seterusnya

Informasi kayak gini sebenarnya sangat berguna buat tim marketing, karena mereka jadi bisa bikin strategi yang beda-beda untuk tiap kelompok pelanggan, bukan dipukul rata semua orang diperlakukan sama.

### 5. Sekarang coba tambah satu fitur lagi: Usia (Age)

Dua fitur tadi sudah cukup informatif, tapi gimana kalau kita tambah fitur **usia**? Mungkin saja pola belanja orang muda dan orang tua itu beda meski income dan spending score-nya mirip. Makanya sekarang kita coba klasterisasi dengan tiga fitur sekaligus: Age, Annual Income, dan Spending Score.

Karena sekarang datanya 3 dimensi, kita gambar dulu scatter plot 3D-nya (belum dikelompokkan, masih data mentah) biar kebayang bentuk sebarannya.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

# Data Age, Annual Income, dan Spending Score
x = df.iloc[:, 2]
y = df.iloc[:, 3]
z = df.iloc[:, 4]

# Membuat plot 3D
fig = plt.figure(figsize=(13, 13))
ax = fig.add_subplot(111, projection='3d')

# Plot data
ax.scatter(x, y, z, c='red', marker='o')

# Label sumbu
ax.set_xlabel('X Axis (Age)')
ax.set_ylabel('Y Axis (Income)')
ax.set_zlabel('Z Axis (Spending Score)')

plt.title('3D Scatter Plot with Matplotlib')
plt.show()

Sama seperti sebelumnya, kita ulangi langkah scaling dan Elbow Method, tapi sekarang pakai tiga fitur sekaligus. Logikanya sama persis, cuma datanya sekarang punya tiga kolom, bukan dua.

In [ ]:
# Menggabungkan data X, Y, Z menjadi array untuk clustering
X = np.array(list(zip(x, y, z)))

# Scaling data dengan MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Menentukan jumlah klaster optimal dengan Elbow Method
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(X_scaled)
    wcss.append(kmeans.inertia_)

# Visualisasi Elbow Method
plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='-')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('WCSS')
plt.grid(True)
plt.show()

Kali ini siku di grafiknya tidak se-tegas sebelumnya, tapi perlambatan penurunannya mulai kelihatan di sekitar **k = 6**. Makanya kita pilih k = 6 untuk kasus 3 fitur ini sedikit lebih banyak dibanding kasus 2 fitur tadi (yang cuma k=5), masuk akal karena menambah dimensi usia bikin ada lebih banyak kombinasi pola yang mungkin terbentuk.

In [ ]:
# Pilih k optimal setelah melihat Elbow
kmeans = KMeans(n_clusters=6, random_state=42)
clusters = kmeans.fit_predict(X)

# Menambahkan hasil klaster ke DataFrame
df['Cluster'] = clusters

# Visualisasi 3D Hasil K-Means
fig = plt.figure(figsize=(13, 13))
ax = fig.add_subplot(111, projection='3d')

# Plot setiap titik dengan warna sesuai klaster
scatter = ax.scatter(x, y, z, c=clusters, cmap='viridis', marker='o')

# Label sumbu
ax.set_xlabel('Age')
ax.set_ylabel('Annual Income (k$)')
ax.set_zlabel('Spending Score (1-100)')

# Menampilkan legenda klaster
legend1 = ax.legend(*scatter.legend_elements(), title="Clusters")
ax.add_artist(legend1)

plt.title('3D Scatter Plot with K-Means Clustering')
plt.show()

Hasil akhirnya kita dapat 6 kelompok pelanggan yang sekarang mempertimbangkan usia juga, bukan cuma income dan spending. Ini membuat segmentasinya jadi lebih detail misalnya pelanggan muda dengan spending tinggi bisa kepisah dari pelanggan yang lebih tua tapi spending-nya juga tinggi, padahal kalau cuma pakai 2 fitur tadi mungkin mereka bakal masuk klaster yang sama.

### Kesimpulan

Satu hal yang perlu saya liat dari modul ini: di bagian 2 fitur, modelnya sebenarnya dilatih pakai `kmeans.fit(X)` yaitu data yang **belum** di-scale padahal proses scaling sudah dilakukan sebelumnya (`X_scaled`). Ini sedikit tidak konsisten, karena harusnya kalau sudah capek-capek melakukan scaling, ya dipakai juga datanya yang sudah di-scale, bukan yang asli. Saya tetap mengikuti kode modul apa adanya di sini supaya hasilnya sama kayak yang ada di PDF, tapi kalau mau versi yang lebih "benar" secara teknis (pakai `X_scaled` pas fit model), tinggal bilang saja, saya bisa buatkan.